# 120 Years of Olympic History

Grasso Gianni & Ismael Trentin

## Dataset

This is a historical dataset on the modern Olympic Games, including all the Games from Athens 1896 to Rio 2016. The data was scraped from www.sports-reference.com in May 2018. More info can be found [here](https://www.kaggle.com/datasets/heesoo37/120-years-of-olympic-history-athletes-and-results).


## Code

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# pio.renderers.default = "notebook"
from plotly.subplots import make_subplots

In [ ]:
df = pd.read_csv('data/athlete_events.csv')

In [ ]:
df.info()

In [ ]:
# drop na values
medals = df[df["Medal"].notna()]

# for each NOC count each Medal type and flatmap that using reset_index
top_noc = (medals.groupby(["NOC", "Medal"])
                 .size()
                 .reset_index(name="count"))
# count NOC occs (auto sort) and take first 20 (just NOCs)
top20 = medals["NOC"].value_counts().head(20).index
top_noc = top_noc[top_noc["NOC"].isin(top20)]

noc_order = medals["NOC"].value_counts().head(20).index.tolist()

fig1 = px.bar(
    top_noc,
    x="NOC", y="count",
    color="Medal",
    color_discrete_map={"Gold": "#FFD700", "Silver": "#C0C0C0", "Bronze": "#CD7F32"},
    category_orders={
      "Medal": ["Gold", "Silver", "Bronze"],
      "NOC": noc_order,
    },
    title="Top 20 Nations by Medal",
    labels={"count": "Medals", "NOC": "Nation"},
)
fig1.show()

In [ ]:
part = (df.groupby(["Year", "Sex"])
          .size()
          .reset_index(name="count"))

total_year = part.groupby("Year")["count"].sum().reset_index(name="total")
years = total_year.sort_values("Year")["Year"].tolist()

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.38, 0.62],
    shared_yaxes=True,
    horizontal_spacing=0.04
)

total_vals = total_year.set_index("Year").reindex(years)
fig.add_trace(
    go.Bar(
        y=years,
        x=total_vals["total"],
        orientation="h",
        marker_color="#6272a4",
        showlegend=False,
        name="Total",
        hovertemplate="Year %{y}: %{x} athletes<extra></extra>"
    ),
    row=1, col=1
)

color_sex = {"M": "#4C72B0", "F": "#DD8452"}
sex_order = ["M", "F"]
for sex in sex_order:
    sub = part[part["Sex"] == sex].set_index("Year")
    total_dict = total_year.set_index("Year")["total"]
    pct = [(sub.loc[yr, "count"] / total_dict[yr] * 100
            if yr in sub.index else 0)
           for yr in years]

    fig.add_trace(
        go.Bar(
            y=years,
            x=pct,
            orientation="h",
            name="Men" if sex == "M" else "Women",
            marker_color=color_sex[sex],
            hovertemplate=f"{'Men' if sex == 'M' else 'Women'}: %{{x:.1f}}%<extra></extra>"
        ),
        row=1, col=2
    )

fig.update_layout(
    title=dict(text="Participants by Year and Sex", font_size=26),
    barmode="stack",
    template="plotly_white",
    height=680,
    legend=dict(title="Sex", traceorder="normal",
                x=1.01, y=0.5, xanchor="left"),
    margin=dict(l=20, r=120, t=80, b=40),
    annotations=[
        dict(text="Total Athletes", x=0.17, y=1.04,
             xref="paper", yref="paper", showarrow=False, font_size=13),
        dict(text="Sex distribution", x=0.72, y=1.04,
             xref="paper", yref="paper", showarrow=False, font_size=13),
    ]
)

fig.update_xaxes(title_text="Athletes [#]", row=1, col=1)
fig.update_xaxes(title_text="Sex [%]", range=[0, 100], row=1, col=2)
fig.update_yaxes(title_text="Year", tickmode="linear", dtick=4, row=1, col=1)

fig.show()

In [ ]:
eta = df.groupby('Year')['Age'].mean().reset_index()

fig = px.line(
    x=eta['Year'],
    y=eta['Age'],
    title='Average Athletes\' Age by Time',
    labels={
      'x': 'Year',
      'y': 'Average age'
    },
    range_x=[df['Year'].min()-1, df['Year'].max()+1],
    range_y=[eta['Age'].min() - 1, eta['Age'].max() + 2],
    markers=True
)

fig.update_layout(
    xaxis=dict(dtick=8),
)

fig.show()